In [2]:
# import pandas as pd
# import numpy as np
# import openpyxl 
# import os

# # --- تنظیمات آدرس‌ها ---
# file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
# output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output2.xlsx'

# # سنسورهای هدف
# target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

# def run_analysis():
#     if not os.path.exists(file_path):
#         print(f"❌ خطا: فایل یافت نشد در مسیر: {file_path}")
#         return

#     try:
#         df = pd.read_excel(file_path, parse_dates=['date'])
#         df.set_index('date', inplace=True)
#         print("✅ مرحله ۱: فایل بارگذاری شد.")
#     except Exception as e:
#         print(f"❌ خطا در خواندن اکسل: {e}")
#         return

#     # ۱. جداسازی بازه سلامت و بازه یک ماه اخیر (Fault)
#     try:
#         last_date = df.index.max()
#         split_date = last_date - pd.Timedelta(days=30)
#         baseline_start = split_date - pd.Timedelta(days=30)

#         df_baseline = df.loc[baseline_start:split_date].copy()
#         df_fault = df.loc[split_date:last_date].copy()
#     except Exception as e:
#         print(f"❌ خطا در پردازش تاریخ‌ها: {e}")
#         return

#     # ۲. محاسبه شیب و TTT برای یک ماه اخیر
#     print("⏳ در حال محاسبه شیب تغییرات و زمان تخمینی برای سنسورهای هدف...")

#     target_analysis_results = []

#     for col in target_sensors:
#         if col not in df_fault.columns: continue

#         try:
#             # الف) حد آستانه از بازه سلامت
#             mean_base = df_baseline[col].mean()
#             std_base = df_baseline[col].std()
#             upper_limit = mean_base + 3 * std_base 

#             # ب) محاسبه شیب (تغییرات در هر ساعت)
#             time_diff_series = df_fault.index.to_series().diff().dt.total_seconds() / 3600.0
#             val_diff = df_fault[col].diff()

#             instant_slopes = val_diff / time_diff_series
#             avg_slope = instant_slopes.mean()

#             current_val = df_fault[col].iloc[-1] 

#             # ج) محاسبه TTT (ساعت مانده تا حد بحرانی)
#             ttt_hours = np.nan
#             if avg_slope > 0 and current_val < upper_limit:
#                 ttt_hours = (upper_limit - current_val) / avg_slope

#             target_analysis_results.append({
#                 'Sensor': col,
#                 'Current_Value': round(current_val, 4),
#                 'Baseline_Limit(3Sigma)': round(upper_limit, 4),
#                 'Average_Slope_per_Hour': round(avg_slope, 6),
#                 'Hours_to_Threshold': round(ttt_hours, 2) if (not np.isnan(ttt_hours) and ttt_hours != np.inf) else "No Risk"
#             })
#         except: continue

#     # ۳. رتبه‌بندی RCA
#     numeric_cols = df_baseline.select_dtypes(include=[np.number]).columns.tolist()
#     rca_list = []
#     for c in numeric_cols:
#         try:
#             dev = abs(df_fault[c].mean() - df_baseline[c].mean()) / (df_baseline[c].std() + 1e-6)
#             rca_list.append({'Sensor': c, 'Deviation_Score': round(dev, 4)})
#         except: continue
#     rca_summary = pd.DataFrame(rca_list).sort_values('Deviation_Score', ascending=False)

#     # ۴. ذخیره در اکسل
#     try:
#         os.makedirs(os.path.dirname(output_filename), exist_ok=True)

#         with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
#             pd.DataFrame(target_analysis_results).to_excel(writer, sheet_name='Speed_Analysis', index=False)
#             rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)

#         print(f"🚀 گزارش با موفقیت ساخته شد:\n{output_filename}")
#     except Exception as e:
#         print(f"❌ خطا در ذخیره فایل: {e}")

# # فراخوانی مستقیم تابع برای جلوگیری از خطای NameError
# run_analysis()

In [4]:
import pandas as pd
import numpy as np
import os
from sklearn.cluster import DBSCAN

# --- تنظیمات آدرس‌ها ---
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output3.xlsx'

target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

def remove_outliers_dbscan(series):
    """حذف داده‌های پرت با استفاده از الگوریتم DBSCAN"""
    if series.empty: return series
    # تغییر شکل داده برای DBSCAN
    X = series.values.reshape(-1, 1)
    # تنظیم پارامترها: eps فاصله همسایگی و min_samples حداقل تعداد همسایه
    dbscan = DBSCAN(eps=series.std()*0.5, min_samples=5)
    clusters = dbscan.fit_predict(X)
    # فقط داده‌هایی که جزو کلاستر اصلی هستند (-1 نباشند)
    return series[clusters != -1]

def run_comprehensive_analysis():
    if not os.path.exists(file_path):
        print(f"❌ فایل یافت نشد.")
        return

    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return

    # ۱. تعریف بازه‌های زمانی
    last_date = df['date'].max()
    fault_start = last_date - pd.Timedelta(days=30)
    baseline_start = fault_start - pd.Timedelta(days=30)

    # جداسازی داده‌ها
    df_baseline = df[(df['date'] >= baseline_start) & (df['date'] < fault_start)].copy()
    df_fault = df[df['date'] >= fault_start].copy()

    # --- بخش اول: آماده‌سازی تب EWMA (داده‌های یک ماه اخیر) ---
    ewma_list = []
    baseline_stats = []

    for col in target_sensors:
        if col not in df.columns: continue

        # الف) محاسبات تب اول (EWMA)
        temp_fault = df_fault[['date', col]].copy()
        temp_fault = temp_fault.rename(columns={col: 'Value'})
        temp_fault['AssetID'] = col
        temp_fault['EWMA'] = temp_fault['Value'].ewm(alpha=0.2, adjust=False).mean()
        ewma_list.append(temp_fault[['date', 'AssetID', 'Value', 'EWMA']])

        # ب) محاسبات تب دوم (Baseline با DBSCAN)
        if col in df_baseline.columns:
            clean_baseline = remove_outliers_dbscan(df_baseline[col].dropna())

            if not clean_baseline.empty:
                mean_val = clean_baseline.mean()
                std_val = clean_baseline.std()

                baseline_stats.append({
                    'AssetID': col,
                    'Clean_Mean': round(mean_val, 4),
                    'Clean_Std': round(std_val, 4),
                    'Normal_Upper_Limit(3Sigma)': round(mean_val + 3*std_val, 4),
                    'Outliers_Removed': len(df_baseline[col]) - len(clean_baseline)
                })

    # --- ذخیره در اکسل با دو تب مجزا ---
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            # تب اول

            if ewma_list:
                df_ewma_final = pd.concat(ewma_list, ignore_index=True)
                df_ewma_final.to_excel(writer, sheet_name='EWMA_Comparison', index=False)

            # تب دوم
            if baseline_stats:
                df_stats_final = pd.DataFrame(baseline_stats)
                df_stats_final.to_excel(writer, sheet_name='Baseline_Stats', index=False)

        print(f"🚀 گزارش با دو تب ساخته شد:\n{output_filename}")
        print("✅ تب Baseline_Stats با موفقیت توسط DBSCAN پاکسازی و محاسبه شد.")
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")

run_comprehensive_analysis()

🚀 گزارش با دو تب ساخته شد:
outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output3.xlsx
✅ تب Baseline_Stats با موفقیت توسط DBSCAN پاکسازی و محاسبه شد.
